In [8]:
# dependencies

from Bio import AlignIO
from Bio import SeqIO
import re
import pandas as pd

In [9]:
# Converting BioCatNet fasta file to dataframe that includes the sequence and the family name

def biocatnet_fasta_to_df(input_fasta):
    data = {'accession': [], 'SFID': [], 'GI': [], 'taxonID': [], 'sequence': []}
    with open(input_fasta) as fp:
        for record in SeqIO.parse(fp, 'fasta'):
            accession = record.id
            sfid = re.findall(r"sfid\|(\d+)", record.id)
            gi = re.findall(r"gi\|(\d+)", record.id)
            taxonID = re.findall(r"taxonID\|(\d+)", record.id)
            sequence = str(record.seq)

            #Attach to dict.
            data['accession'].append(accession)
            data['SFID'].append(", ".join(sfid))
            data['GI'].append(", ".join(gi))
            data['taxonID'].append(", ".join(taxonID))
            data['sequence'].append(sequence)
    df = pd.DataFrame(data)
    return df

In [34]:
# Converting HMM search output for BioCatNet data (e = 0.00001) to FASTA file
input = '/home/alecia/chapter3-repo/output/hmm_search_3/biocatnet/5/ihelixhits_e_0p00001_z__T_.aln'
outfile_fasta = '/home/alecia/chapter3-repo/output/hmm_search_3/biocatnet/5/ihelixhits_e_0p00001_z__T_.fasta'

with open(input) as infile:
    alignment = AlignIO.read(infile, 'stockholm')

SeqIO.write(alignment, outfile_fasta, 'fasta')

113

In [35]:
# Convert FASTA file into df of the results

df_results = biocatnet_fasta_to_df('/home/alecia/chapter3-repo/output/hmm_search_3/biocatnet/5/ihelixhits_e_0p00001_z__T_.fasta')
df_results

,accession,SFID,GI,taxonID,sequence
0,sid|75248|pid|54607|hfid|349|sfid|33|gi|308460...,33,308460667,31234,---DSVCFD-LWVAGMETTSNTLYWSLLYVLL-
1,sid|75248|pid|54607|hfid|349|sfid|33|gi|308460...,33,308460667,31234,---DSVCFD-LWVAGMETTSNTLYWSLLYVLL-
2,sid|6214|pid|4230|hfid|316|sfid|19|gb|AAH56258...,19,33604018,9606,-ENVNQCILEMLIAAPDTMSVSL-FFMLFLIA-
3,sid|6215|pid|4230|hfid|316|sfid|19|gb|AAH20767...,19,116283623,9606,-ENVNQCILEMLIAAPDTMSVSL-FFMLFLIA-
4,sid|6197|pid|4216|hfid|316|sfid|19|gi|53200717...,19,532007179,79684,-ENVNQCILEMLIAAPDTMSVTLYFMLLLI---
...,...,...,...,...,...
108,sid|6185|pid|4208|hfid|316|sfid|19|gi|28118262...,19,281182626,10116,-ENVNQCILEMLIAAPDTMSVTLYVMLLLI---
109,sid|6202|pid|4221|hfid|316|sfid|19|gb|AAP43633...,19,31415709,9337,-ENVNQCILEMLIAAPDTLSVTVYFMLLLI---
110,sid|4935|pid|3317|hfid|1296|sfid|305|gb|AAS134...,305,42398143,6984,--QLVSLCLDLFMAGSETTSNTLGFAVLYMLL-
111,sid|6201|pid|4220|hfid|316|sfid|19|gi|51186851...,19,511868515,9669,-ENVNQCILEMLVAAPDTMSVSV-FFMLFLIA-


In [13]:
# Open database and put into dataframe

df_database = biocatnet_fasta_to_df('/home/alecia/chapter3-repo/input/databases/biocatnet_p450.fasta')
df_database

,accession,SFID,GI,taxonID,sequence
0,sid|90155|pid|67114|hfid|29|sfid|2|gi|55729550...,2,557295506,38654,MDAGTTGILILLLLIFVLCYLVLEINRKRAQLPAGPAPWPVLGNLW...
1,sid|90154|pid|67113|hfid|29|sfid|2|gi|60264309...,2,602643091,176946,GPTPWPFLGNLLQRDVLPVDRLYKKLTAKYGPIFTVWIGSKPMVAL...
2,sid|90156|pid|67115|hfid|29|sfid|2|gi|60267100...,2,602671004,176946,MELTWAGALLLLCIFFTLFSSFQIYKKKGQLPPGPTPWPFLGNLLQ...
3,sid|90236|pid|67189|hfid|29|sfid|2|gb|ETE74148...,2,565323585,8665,MELTWTGVLLLLCVLFILFSSFQMYKKKGQLPPGPTPWPILGNLLQ...
4,sid|101660|pid|77115|hfid|29|sfid|2|gi|6026704...,2,602670425,176946,MELTWAGALLLLSILITLFSSFQMYKKKGQLPPGPTPWPFLGNLLQ...
...,...,...,...,...,...
52669,sid|14140|pid|10364|hfid|1674|sfid|5006|gi|118...,5006,"118351107, 164519821",5911,MSIIQLFIGFIIGTIIYQTVIKTLYLYYVYKRRYGKDIIVLFYPVI...
52670,sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,5009,"164519833, 118380288",5911,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...
52671,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,5060,667644755,655819,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...
52672,sid|85661|pid|63286|hfid|1696|sfid|5280|gi|667...,5280,667652271,655819,MFMLWIAAAVFGLIIIGRPLIVYLQDVKGFRKYPVQNFLSGISPLA...


In [36]:
# Merge results with database to see which sequences are missing from results
## Accession cannot be used to merge on due to the presence of 'sub seq' 

merged_df = (pd.merge(df_database, df_results, how='left', on=['GI', 'SFID', 'taxonID'])
             .fillna(0))
merged_df

,accession_x,SFID,GI,taxonID,sequence_x,accession_y,sequence_y
0,sid|90155|pid|67114|hfid|29|sfid|2|gi|55729550...,2,557295506,38654,MDAGTTGILILLLLIFVLCYLVLEINRKRAQLPAGPAPWPVLGNLW...,0,0
1,sid|90154|pid|67113|hfid|29|sfid|2|gi|60264309...,2,602643091,176946,GPTPWPFLGNLLQRDVLPVDRLYKKLTAKYGPIFTVWIGSKPMVAL...,0,0
2,sid|90156|pid|67115|hfid|29|sfid|2|gi|60267100...,2,602671004,176946,MELTWAGALLLLCIFFTLFSSFQIYKKKGQLPPGPTPWPFLGNLLQ...,0,0
3,sid|90236|pid|67189|hfid|29|sfid|2|gb|ETE74148...,2,565323585,8665,MELTWTGVLLLLCVLFILFSSFQMYKKKGQLPPGPTPWPILGNLLQ...,0,0
4,sid|101660|pid|77115|hfid|29|sfid|2|gi|6026704...,2,602670425,176946,MELTWAGALLLLSILITLFSSFQMYKKKGQLPPGPTPWPFLGNLLQ...,0,0
...,...,...,...,...,...,...,...
52672,sid|14140|pid|10364|hfid|1674|sfid|5006|gi|118...,5006,"118351107, 164519821",5911,MSIIQLFIGFIIGTIIYQTVIKTLYLYYVYKRRYGKDIIVLFYPVI...,0,0
52673,sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,5009,"164519833, 118380288",5911,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...,0,0
52674,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,5060,667644755,655819,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...,0,0
52675,sid|85661|pid|63286|hfid|1696|sfid|5280|gi|667...,5280,667652271,655819,MFMLWIAAAVFGLIIIGRPLIVYLQDVKGFRKYPVQNFLSGISPLA...,0,0


In [37]:
# Create new dataframe from where there are no hits

no_hits = merged_df.loc[merged_df["sequence_y"] == 0, ["GI", "accession_x", "sequence_x", "SFID", "taxonID"]]
no_hits

,GI,accession_x,sequence_x,SFID,taxonID
0,557295506,sid|90155|pid|67114|hfid|29|sfid|2|gi|55729550...,MDAGTTGILILLLLIFVLCYLVLEINRKRAQLPAGPAPWPVLGNLW...,2,38654
1,602643091,sid|90154|pid|67113|hfid|29|sfid|2|gi|60264309...,GPTPWPFLGNLLQRDVLPVDRLYKKLTAKYGPIFTVWIGSKPMVAL...,2,176946
2,602671004,sid|90156|pid|67115|hfid|29|sfid|2|gi|60267100...,MELTWAGALLLLCIFFTLFSSFQIYKKKGQLPPGPTPWPFLGNLLQ...,2,176946
3,565323585,sid|90236|pid|67189|hfid|29|sfid|2|gb|ETE74148...,MELTWTGVLLLLCVLFILFSSFQMYKKKGQLPPGPTPWPILGNLLQ...,2,8665
4,602670425,sid|101660|pid|77115|hfid|29|sfid|2|gi|6026704...,MELTWAGALLLLSILITLFSSFQMYKKKGQLPPGPTPWPFLGNLLQ...,2,176946
...,...,...,...,...,...
52672,"118351107, 164519821",sid|14140|pid|10364|hfid|1674|sfid|5006|gi|118...,MSIIQLFIGFIIGTIIYQTVIKTLYLYYVYKRRYGKDIIVLFYPVI...,5006,5911
52673,"164519833, 118380288",sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...,5009,5911
52674,667644755,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...,5060,655819
52675,667652271,sid|85661|pid|63286|hfid|1696|sfid|5280|gi|667...,MFMLWIAAAVFGLIIIGRPLIVYLQDVKGFRKYPVQNFLSGISPLA...,5280,655819


In [38]:
# Family counts in results and no hits

families_hits = (df_results.value_counts(df_results['SFID'])
                 .to_csv('/home/alecia/chapter3-repo/analysis/round3_missing_biocatnet/family_count_results_r3/round3_biocatnet_hits_families_0p00001.txt')
)

families_nohits = (no_hits.value_counts(no_hits['SFID'])
                   .to_csv('/home/alecia/chapter3-repo/analysis/round3_missing_biocatnet/family_count_results_r3/round3_biocatnet_nohits_families_0p00001.txt')
)